|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Reading through the table<h1>|
|<h2>Lecture:</h2>|<h1><b>The same arithmetic, with the keys scattered<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root. The directory you start from does not matter.
import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Attention that reads through the page table

The arithmetic does not change. The addresses change.

`keys[seq, kv_head, pos]` becomes a block table, a pool, and a lookup:

    physical = block_table[seq][pos // block_size]
    entry    = pool[physical][kvh][pos % block_size]

In [2]:
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

num_seqs, num_heads, num_kv_heads, head_dim, context_len = 4, 8, 2, 64, 100
BLOCK_SIZE = 16

keys = torch.randn(num_seqs, num_kv_heads, context_len, head_dim, device=device)
values = torch.randn(num_seqs, num_kv_heads, context_len, head_dim, device=device)
query = torch.randn(num_seqs, num_heads, head_dim, device=device)
print(f'{num_seqs} sequences, {context_len} tokens each, blocks of {BLOCK_SIZE}')

4 sequences, 100 tokens each, blocks of 16


### Scatter the cache, on purpose

A real pool holds blocks in no order. The allocator gives out blocks as they
become free, so the blocks of one sequence sit anywhere.

Build the pool that way on purpose. A gather that assumes an order passes a
tidy test and then fails in production.

In [3]:
from tests.helpers import build_paged

key_cache, value_cache, block_tables, context_lens = build_paged(keys, values, BLOCK_SIZE)

print(f'pool:         {tuple(key_cache.shape)}   (blocks, kv_heads, block_size, head_dim)')
print(f'block table:  {tuple(block_tables.shape)}')
print(f'\nsequence 0 lives in blocks: {block_tables[0].tolist()}')
print(f'sequence 1 lives in blocks: {block_tables[1].tolist()}')
print('\nnot contiguous, not sorted, not adjacent. That is the point.')

pool:         (28, 2, 16, 64)   (blocks, kv_heads, block_size, head_dim)
block table:  (4, 7)

sequence 0 lives in blocks: [3, 14, 10, 17, 5, 0, 7]
sequence 1 lives in blocks: [21, 19, 18, 23, 25, 4, 2]

not contiguous, not sorted, not adjacent. That is the point.


### Gather first, then do the same arithmetic

In [4]:
import math

def paged_attention(query, key_cache, value_cache, block_tables, context_lens, scale=None):
  num_seqs, num_heads, head_dim = query.shape
  num_kv_heads, block_size = key_cache.shape[1], key_cache.shape[2]
  group = num_heads // num_kv_heads
  scale = scale or 1.0/math.sqrt(head_dim)
  output = torch.empty_like(query)

  for seq in range(num_seqs):
    context_len = int(context_lens[seq])
    blocks = block_tables[seq, : (context_len + block_size - 1)//block_size].long()
    # (blocks, kv_heads, block_size, head_dim) -> (kv_heads, tokens, head_dim), then cut to context_len
    seq_keys = key_cache[blocks].permute(1,0,2,3).reshape(num_kv_heads, -1, head_dim)[:, :context_len]
    seq_values = value_cache[blocks].permute(1,0,2,3).reshape(num_kv_heads, -1, head_dim)[:, :context_len]
    for head in range(num_heads):
      kv_head = head // group
      scores = (seq_keys[kv_head] @ query[seq,head]) * scale
      output[seq,head] = torch.softmax(scores, dim=0) @ seq_values[kv_head]
  return output

output = paged_attention(query, key_cache, value_cache, block_tables, context_lens)
print('output:', tuple(output.shape))

output: (4, 8, 64)


In [5]:
# the oracle: the same arithmetic on the ORIGINAL contiguous tensors
def reference_attention(query, keys, values):
  num_seqs, num_heads, head_dim = query.shape
  group = num_heads // keys.shape[1]
  output = torch.empty_like(query)
  for seq in range(num_seqs):
    for head in range(num_heads):
      scores = (keys[seq, head//group] @ query[seq,head]) / math.sqrt(head_dim)
      output[seq,head] = torch.softmax(scores, dim=0) @ values[seq, head//group]
  return output

expected = reference_attention(query, keys, values)
print('max difference:', (output - expected).abs().max().item())
print('\nthe gather found every byte, in the right order, from a shuffled pool')

max difference: 0.0

the gather found every byte, in the right order, from a shuffled pool


### The masking trap

The allocator recycles blocks. A slot past `context_len` holds whatever the
last sequence left there. Score that slot and you get a plausible, wrong
answer.

In [6]:
poisoned_keys, poisoned_values = key_cache.clone(), value_cache.clone()
for seq in range(num_seqs):
  for block in range(block_tables.shape[1]):
    block_id = int(block_tables[seq,block])
    for offset in range(BLOCK_SIZE):
      if block*BLOCK_SIZE + offset >= context_len:
        poisoned_keys[block_id,:,offset] = 999.0       # someone else's tokens
        poisoned_values[block_id,:,offset] = 999.0

poisoned_output = paged_attention(query, poisoned_keys, poisoned_values, block_tables, context_lens)
print('unchanged after poisoning the unused slots:',
      torch.allclose(output, poisoned_output, atol=1e-5))
print('\n(this passes because the gather cuts to n. Forget that and it will not.)')

unchanged after poisoning the unused slots: True

(this passes because the gather cuts to n. Forget that and it will not.)


# What the gather cost you

In [7]:
if device == 'cuda':
  num_seqs, num_heads, num_kv_heads, head_dim, context_len = 32, 16, 8, 128, 512
  keys = torch.randn(num_seqs, num_kv_heads, context_len, head_dim, device=device, dtype=torch.float16)
  values = torch.randn(num_seqs, num_kv_heads, context_len, head_dim, device=device, dtype=torch.float16)
  query = torch.randn(num_seqs, num_heads, head_dim, device=device, dtype=torch.float16)
  key_cache, value_cache, bench_block_tables, bench_context_lens = build_paged(keys, values, BLOCK_SIZE)

  paged = cudalib.bench_ms(lambda: paged_attention(query, key_cache, value_cache, bench_block_tables, bench_context_lens), iters=5, warmup=2)
  dense = cudalib.bench_ms(lambda: F.scaled_dot_product_attention(
            query.unsqueeze(2), keys.repeat_interleave(num_heads//num_kv_heads,1), values.repeat_interleave(num_heads//num_kv_heads,1)),
            iters=20, warmup=5)

  print(f'contiguous SDPA:  {dense:8.3f} ms')
  print(f'paged, in python: {paged:8.3f} ms')
  print(f'\nyou just paid {paged/dense:.0f}x for the memory you saved')

contiguous SDPA:     0.969 ms
paged, in python:   17.645 ms

you just paid 18x for the memory you saved


### That number is the bill for stage 06

Paging gave you about ten times the resident sequences. It cost about the same
factor in attention speed.

The code above explains the cost. It is a **Python loop over sequences**. Each
pass does one gather and one matmul.

That sentence holds two separate problems:

- One kernel launch per sequence per head. There should be one launch.
- The gather **materialises** K and V. Every byte travels to HBM and comes
  back before anything reads it.

One kernel removes both problems. The second fix is the interesting one. The
score row never has to exist, if you fold each tile into a running softmax.

This function now becomes the oracle. You check every later kernel against
it.

    ./vc guide 8